# Lab 1B: errors, retries, and limits
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 1 · about 20 minutes**

**Objectives**
- Recognize the common API error types and which ones are retryable
- Implement exponential backoff for transient errors only
- Avoid stacked retry layers
- Watch token usage grow with history, and predict stop_reason values

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic requests
import os, json, time, random, importlib
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings. Code imports from here; nothing is hard-coded elsewhere."""
import os

# Pin the exact model ID your tests passed on. Confirm current IDs on the models overview page:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
MAX_TOKENS = 1024
MAX_RETRIES = 2        # one retry layer: the SDK's own
TIMEOUT_S = 60
API_VERSION = "2023-06-01"
# The key is NEVER stored here. It comes from the environment: ANTHROPIC_API_KEY

In [ ]:
import config
importlib.reload(config)          # pick up edits if you rerun the cell above
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    """Print the parts of a response you check on every call."""
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

print("model:", config.MODEL, "| max_retries:", config.MAX_RETRIES, "| timeout:", config.TIMEOUT_S, "s")

## Drill 1: read the errors
Fill in `PREDICT` with the HTTP status you expect for each case, then run.

In [ ]:
PREDICT = {"bad_key": "?", "bad_model": "?", "empty_messages": "?", "zero_max_tokens": "?"}
RETRYABLE = {408, 409, 429, 500, 529}     # transient: back off and retry. Everything else: fix the request.

def attempt(label, fn):
    try:
        fn(); print(f"{label:16s} OK")
    except anthropic.APIStatusError as e:
        err = (e.body or {}).get("error", {}) if isinstance(e.body, dict) else {}
        print(f"{label:16s} predicted={PREDICT[label]:>4} actual={e.status_code} type={err.get('type')} "
              f"retry={'yes' if e.status_code in RETRYABLE else 'NO, fix the request'}")
        print(f"{'':16s} {str(err.get('message', e))[:110]}")

bad_client = anthropic.Anthropic(api_key="sk-ant-this-key-is-invalid", max_retries=0)
attempt("bad_key", lambda: bad_client.messages.create(model=config.MODEL, max_tokens=50,
        messages=[{"role": "user", "content": "hi"}]))
attempt("bad_model", lambda: client.messages.create(model="claude-model-that-does-not-exist", max_tokens=50,
        messages=[{"role": "user", "content": "hi"}]))
attempt("empty_messages", lambda: client.messages.create(model=config.MODEL, max_tokens=50, messages=[]))
attempt("zero_max_tokens", lambda: client.messages.create(model=config.MODEL, max_tokens=0,
        messages=[{"role": "user", "content": "hi"}]))

The SDK raises typed exceptions you can catch directly, for example `anthropic.AuthenticationError` (401), `anthropic.NotFoundError` (404), `anthropic.BadRequestError` (400), and `anthropic.RateLimitError` (429). All of them are `anthropic.APIStatusError` with a `status_code`.

## Drill 2: retry with backoff, for transient errors only
You cannot trigger real 429 or 529 errors on demand, so a small simulator raises them in a scripted order.

In [ ]:
class SimulatedStatusError(Exception):
    def __init__(self, status_code, retry_after=None):
        super().__init__(f"simulated HTTP {status_code}")
        self.status_code, self.retry_after = status_code, retry_after

def scripted(outcomes):
    """Return a fake API call that raises the scripted statuses, then succeeds."""
    it = iter(outcomes)
    calls = {"n": 0}
    def call():
        calls["n"] += 1
        status = next(it, "ok")
        if status != "ok":
            raise SimulatedStatusError(status, retry_after=1 if status == 429 else None)
        return "success"
    return call, calls

def with_backoff(fn, max_attempts=4, base=0.5, cap=8.0):
    """Exponential backoff with full jitter. Retries transient statuses only; honors retry-after when given."""
    for attempt_no in range(1, max_attempts + 1):
        try:
            return fn()
        except Exception as e:
            status = getattr(e, "status_code", None)
            if status not in RETRYABLE or attempt_no == max_attempts:
                print(f"  attempt {attempt_no}: HTTP {status} -> giving up"); raise
            wait = getattr(e, "retry_after", None) or random.uniform(0, min(cap, base * 2 ** attempt_no))
            print(f"  attempt {attempt_no}: HTTP {status} -> retry in {wait:.2f}s")
            time.sleep(wait)

In [ ]:
print("Case 1: overloaded, then rate limited, then success")
fn, calls = scripted([529, 429]); print("  result:", with_backoff(fn), "| calls:", calls["n"])

print("\nCase 2: a 400 must not be retried")
fn, calls = scripted([400])
try:
    with_backoff(fn)
except SimulatedStatusError:
    print("  stopped after", calls["n"], "call: fix the request instead")

### Stacked retries multiply traffic
The SDK already retries transient failures (it defaults to 2 retries, so up to 3 attempts). Wrapping that in your own 5-attempt loop means one user click can become 15 requests during an outage.

In [ ]:
def worst_case_requests(custom_attempts, sdk_max_retries):
    return custom_attempts * (sdk_max_retries + 1)
for custom, sdk in [(1, 2), (5, 2), (5, 0), (3, 5)]:
    print(f"custom loop attempts={custom}, SDK max_retries={sdk} -> up to {worst_case_requests(custom, sdk)} requests per click")

Pick ONE retry layer. Either keep the SDK's (tune `max_retries` in `config.py`), or set `anthropic.Anthropic(max_retries=0)` and own the backoff yourself.

## Drill 3: tokens grow with history
Every call resends the full conversation, so input tokens climb each turn. `count_tokens` measures a request before you send it.

In [ ]:
turns = ["Hi, I need help with my bill.", "My account is C-1001.", "The September bill was 74.99 USD.",
         "It was charged twice on 1 September.", "I paid by card ending 4421.", "Please summarize my case."]
history, rows = [], []
for i, text in enumerate(turns, 1):
    history.append({"role": "user", "content": text})
    predicted = client.messages.count_tokens(model=config.MODEL, messages=history).input_tokens
    resp = client.messages.create(model=config.MODEL, max_tokens=150, messages=history)
    history.append({"role": "assistant", "content": resp.content})
    rows.append((i, predicted, resp.usage.input_tokens, resp.usage.output_tokens))
    print(f"turn {i}: count_tokens={predicted:5d} actual_input={resp.usage.input_tokens:5d} output={resp.usage.output_tokens:4d}")

In [ ]:
# Optional cost estimate: copy current per-million-token prices for your model from the official pricing page.
PRICE_IN_PER_MTOK = None    # for example 3.00 (USD)
PRICE_OUT_PER_MTOK = None
if PRICE_IN_PER_MTOK and PRICE_OUT_PER_MTOK:
    total = sum(r[2] * PRICE_IN_PER_MTOK + r[3] * PRICE_OUT_PER_MTOK for r in rows) / 1_000_000
    print(f"estimated cost of this chat: ${total:.5f}")
else:
    print("Fill in both prices to estimate cost. Notice which grows faster: input or output tokens.")

## Drill 4: predict the stop_reason
Write `end_turn`, `max_tokens`, `stop_sequence`, or `tool_use` for each request, then run.

In [ ]:
PREDICT_STOP = {"a": "?", "b": "?", "c": "?", "d": "?"}
lookup_tool = {"name": "lookup_ticket", "description": "Look up a support ticket by ID (format T-12).",
               "input_schema": {"type": "object", "properties": {"ticket_id": {"type": "string"}}, "required": ["ticket_id"]}}
a = client.messages.create(model=config.MODEL, max_tokens=200,
        messages=[{"role": "user", "content": "Say hello in five words."}])
b = client.messages.create(model=config.MODEL, max_tokens=15,
        messages=[{"role": "user", "content": "Explain how fiber broadband works, in detail."}])
c = client.messages.create(model=config.MODEL, max_tokens=200, stop_sequences=["END"],
        messages=[{"role": "user", "content": "List two router fixes, then write END, then a third fix."}])
d = client.messages.create(model=config.MODEL, max_tokens=300, tools=[lookup_tool],
        messages=[{"role": "user", "content": "What is the status of ticket T-77?"}])
for k, r in {"a": a, "b": b, "c": c, "d": d}.items():
    print(f"{k}: predicted={PREDICT_STOP[k]:14s} actual={r.stop_reason:14s} stop_sequence={getattr(r, 'stop_sequence', None)}")

Request **d** returns `tool_use`: Claude is asking your code to run `lookup_ticket`. Handling that properly is Day 3.

## You should now have
- [ ] A table of error types and whether each is retryable
- [ ] `with_backoff()` that retries transient errors only
- [ ] A single retry layer decision recorded in `config.py`
- [ ] A feel for how history drives input tokens and cost

## Check yourself
1. Which of 400, 401, 404, 429, and 529 should be retried?
2. Your service already uses the SDK's retries. Why is a second retry loop risky?
3. Which stop_reason means your code must run a tool and call again?